# Modeling: pronostico de rendimiento de maiz

`yield` es continuo (t/ha), por lo que se comparan regresores con MAE, RMSE y R2. Accuracy/F1 corresponden a clasificacion y no aplican aqui. La logica de entrenamiento y evaluacion esta en `src/maize_forecasting/modeling.py`; este notebook muestra el flujo y los resultados.

## Protocolo

Se usa `model_input_candidates_cutoff_50` y las features seleccionadas en Feature Engineering. Se comparan Ridge, Random Forest, Extra Trees, Gradient Boosting y HistGradientBoosting, junto con un baseline de mediana. La seleccion usa macro-MAE OOF de cuatro folds rolling-origin; cada fold valida campañas posteriores a las de entrenamiento. Despues se ajusta el candidato ganador con campañas 2003-2018.

El periodo 2019-2022 se informa como holdout exploratorio: ya habia sido consultado en iteraciones previas, asi que no es un test final independiente. No se afinan hiperparametros ni se afirma generalizacion a unidades administrativas nunca vistas.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").is_dir():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from maize_forecasting.modeling import run_modeling
from maize_forecasting.star_schema import query_star

## Comparacion y ajuste

`overwrite=True` reemplaza el artefacto generado previamente. El modelo se selecciona por macro-MAE OOF y el baseline se calcula dentro de cada fold usando solo sus datos de entrenamiento.

In [ ]:
modeling_result = run_modeling(overwrite=True)
modeling_report = modeling_result["report"]
print("Estado:", modeling_report["status"])
print("Regresor elegido:", modeling_report["selection"]["selected_model"])
print("Gana al baseline en rolling OOF:", modeling_report["selection"]["candidate_beats_dummy"])
print("Artefacto:", modeling_report["model_artifact"])

## Resultados OOF y holdout exploratorio

Las predicciones OOF combinan exclusivamente folds temporales del periodo de desarrollo. El holdout se usa para una comprobacion exploratoria del regresor elegido y del baseline; no para seleccionar algoritmos.

In [ ]:
oof_results = query_star(
    """
    SELECT model, n, n_seasons, macro_season_mae, mae, rmse, r2
    FROM model_comparison_results
    WHERE split = 'rolling_origin_oof'
    ORDER BY macro_season_mae
    """
)
display(oof_results)

holdout_results = pd.DataFrame([
    {"model": modeling_report["selection"]["selected_model"], **modeling_report["exploratory_holdout_metrics"]},
    {"model": "dummy_median", **modeling_report["exploratory_holdout_dummy_baseline"]},
])
display(holdout_results[["model", "n", "n_seasons", "macro_season_mae", "mae", "rmse", "r2"]])

## Limites y recibo

La metrica primaria es macro-MAE por campaña; no se usa MAPE porque hay rendimientos cero. El artefacto es experimental, entrenado solo hasta 2018, con cutoff 50%. El holdout 2019-2022 no es independiente y la transferencia a regiones fuera de entrenamiento no se midio. Las features añadidas despues de la evaluacion (AWC, coordenadas, drenaje) no entran en este modelo.

In [ ]:
print(f"Target rows del candidato: {modeling_report['exploratory_holdout_metrics']['n']:,} en holdout exploratorio")
print(f"Features: {len(modeling_report['feature_names'])}")
print(f"Entrenamiento final incluye temporadas hasta 2018")
print(f"Artefacto guardado: {Path(modeling_report['model_artifact']).is_file()}")
print(f"No se ajustaron hiperparametros: {not modeling_report['hyperparameter_tuning_performed']}")